# Exercise 3: Building a Graph-Based Workflow

In Exercises 1 and 2 you built and chatted with individual host agents. In this exercise you will **wire those agents into a directed graph** — a workflow where each stage automatically feeds the next, and the framework handles routing, parallelism, and output collection for you.

By the end of this exercise you will know how to:

| Concept | What you'll do |
|---|---|
| `Executor` + `@handler` | Define a processing node with a typed message handler |
| `@executor` decorator | Create a lightweight function-based executor |
| `WorkflowBuilder` | Assemble executors into a directed graph |
| `add_edge()` | Define how messages flow between nodes |
| `workflow.run()` | Execute the graph and collect results |
| `AgentExecutor` | Drop an AI agent into the graph as a first-class node |
| Conditional edges | Route messages down different paths at runtime |

> **Kernel reminder** — make sure you're running the **Workshop (Python 3.12)** kernel. Open the kernel picker (top-right of the notebook in VS Code) and select it under *Jupyter Kernels*. If you don't see it, run **Python: Select Interpreter** from the Command Palette (`Ctrl+Shift+P`) and pick the interpreter at `/opt/venv/bin/python` first.

In [ ]:
from pathlib import Path
from typing import Any, Never
from dataclasses import dataclass

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    handler,
)

from utils import load_env, create_agent, AgentOptions, move_to_repo_root

move_to_repo_root()
load_env()  # reads .env at the repo root

---
## 1. Workflows vs. agents

An **agent** is LLM-driven: you hand it a message and it decides — dynamically, turn by turn — which tools to call and what to say next.

A **workflow** is graph-driven: you explicitly define the nodes (executors) and the connections between them (edges). The framework then executes those nodes in order, passing the output of each one as the input to the next. The graph topology is fixed at build time; what changes at runtime is only the data flowing through it.

```
Agent                          Workflow
──────────────────────         ──────────────────────────────────────
           ┌───────┐           TopicFormatter  ──▶  Researcher
 message ─▶│  LLM  │─▶ reply       │                    │
           │ tools │               ▼                    ▼
           └───────┘           MaraExecutor ◀── HostBridgeExecutor
   (dynamic, open-ended)       (fixed graph, deterministic routing)
```

Workflows shine when:
- The steps are **known in advance** (research → host prep → episode brief)
- You need **parallel execution** (both hosts run at the same time)
- You need **conditional branching** (plot topics go to Mara, character topics go to Dev)
- You want **type-checked message passing** between stages

---
## 2. The core building block: Executors

An **executor** is a single node in the workflow graph. It receives a typed message, does some work, and either:
- sends a new message downstream with `ctx.send_message(...)`, or
- yields a final result to the caller with `ctx.yield_output(...)`.

There are two ways to create an executor:

### Class-based — for executors that need state or multiple handlers

```python
class MyExecutor(Executor):
    @handler
    async def my_handler(self, message: InputType, ctx: WorkflowContext[OutputType]) -> None:
        await ctx.send_message(...)   # to the next node
        # OR
        await ctx.yield_output(...)  # as a workflow result
```

### Function-based — for stateless single-handler executors

```python
@executor(id="my_executor")
async def my_executor(message: InputType, ctx: WorkflowContext[OutputType]) -> None:
    await ctx.send_message(...)
```

The `WorkflowContext` type parameter tells the framework what type of message this handler produces:
- `WorkflowContext[str]` — sends a `str` to the next node
- `WorkflowContext[Never, str]` — only yields output to the caller (no downstream message)
- `WorkflowContext` — no output (side-effects only, e.g. logging)

In [ ]:
# --- Class-based executor ---
class TopicFormatterExecutor(Executor):
    """Formats a raw topic string into a structured research prompt."""

    @handler
    async def format_topic(self, topic: str, ctx: WorkflowContext[str]) -> None:
        prompt = (
            f"Research question for a Lost podcast episode: {topic}\n"
            "Provide 2-3 paragraphs of relevant background from the show."
        )
        await ctx.send_message(prompt)


# --- Function-based executor ---
@executor(id="topic_printer")
async def topic_printer(text: str, ctx: WorkflowContext[Never, str]) -> None:
    """Prints the text and yields it as the workflow's final output."""
    print(text)
    await ctx.yield_output(text)

---
## 3. Wiring executors together: `WorkflowBuilder`

`WorkflowBuilder` assembles executors into a graph. You always:
1. Pass the **starting executor** to the constructor
2. Call `add_edge(source, target)` for each connection
3. Call `build()` to get a runnable `Workflow`

Run the workflow with `await workflow.run(input)`. The input must match the type expected by the start executor's handler.

Collect results with `events.get_outputs()` — it returns a list of everything passed to `ctx.yield_output(...)` anywhere in the graph.

In [ ]:
# Class-based executors must be instantiated; function-based executors already are
formatter = TopicFormatterExecutor()

# Build a 2-node linear workflow: formatter → printer
first_workflow = (
    WorkflowBuilder(start_executor=formatter)
    .add_edge(formatter, topic_printer)
    .build()
)

# Run it — the string goes into the formatter, which sends a prompt to the printer,
# which yields it as output.
events = await first_workflow.run("The hatch — what was really inside it?")

print("\nWorkflow outputs:")
for output in events.get_outputs():
    print(output)

---
## 4. Streaming execution

Pass `stream=True` to `workflow.run()` to receive events as they are emitted rather than waiting for the whole workflow to finish. This is especially useful when the workflow contains AI agents that produce text incrementally.

Each event has a `type` field:
- `"output"` — a final result from `ctx.yield_output(...)`
- `"intermediate"` — a progress event (requires `intermediate_output_from` on the builder)

The streaming and non-streaming modes produce identical *results* — only the timing is different.

In [ ]:
# Streaming version of the same workflow
async for event in first_workflow.run("Why can't anyone leave the island?", stream=True):
    if event.type == "output":
        print(f"[output] {event.data}")

---
## 5. Agents in the graph: `AgentExecutor`

`AgentExecutor` wraps any agent created by `create_agent()` and turns it into a workflow node. The node always receives an `AgentExecutorRequest` and produces an `AgentExecutorResponse`.

```python
AgentExecutor(agent, id="unique_id")
```

The `id` is how the framework identifies this node in logs and events.

Because `AgentExecutor` expects `AgentExecutorRequest` as its input type, you'll often need a small **bridge executor** to convert between plain data and the agent's required format:

```python
AgentExecutorRequest(
    messages=[Message(role="user", contents=["your prompt here"])],
    should_respond=True,
)
```

To extract the agent's response text from the output:
```python
response.agent_run_response.text
```

In [ ]:
# Load the host definitions created in Exercise 1
mara_instructions = Path("output/agents/host-mara-finch.md").read_text()
dev_instructions  = Path("output/agents/host-dev-navarro.md").read_text()

# Wrap each host agent in an AgentExecutor — they become nodes in the graph
mara_executor = AgentExecutor(
    create_agent(AgentOptions(name="Mara Finch", instructions=mara_instructions)),
    id="mara_finch",
)
dev_executor = AgentExecutor(
    create_agent(AgentOptions(name="Dev Navarro", instructions=dev_instructions)),
    id="dev_navarro",
)

# A shared terminal executor that extracts the agent response text and yields it as output
@executor(id="response_printer")
async def response_printer(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Yields the agent's response text as workflow output."""
    await ctx.yield_output(response.agent_run_response.text)

In [ ]:
# Minimal single-agent workflow: mara_executor → response_printer
single_agent_workflow = (
    WorkflowBuilder(start_executor=mara_executor)
    .add_edge(mara_executor, response_printer)
    .build()
)

# AgentExecutors always receive an AgentExecutorRequest as their first input
request = AgentExecutorRequest(
    messages=[Message(role="user", contents=["What's your honest take on the polar bear?"])],
    should_respond=True,
)

events = await single_agent_workflow.run(request)
for output in events.get_outputs():
    print(output)

---
## 6. A podcast research pipeline

Now let's build a proper multi-stage pipeline. A realistic pre-production workflow for a podcast episode looks like this:

```
topic (str)
    │
    ▼
topic_to_request        # bridge: str → AgentExecutorRequest
    │
    ▼
research_executor       # AI agent: researches the topic
    │
    ▼
research_to_host        # bridge: AgentExecutorResponse → AgentExecutorRequest (with context)
    │
    ▼
mara_executor           # Mara forms her angle on the research
    │
    ▼
response_printer        # terminal: yield output
```

The two **bridge executors** are the glue. Each converts between the output type of its predecessor and the input type of its successor, keeping each main node focused on a single responsibility.

In [ ]:
# A generic research agent
research_executor = AgentExecutor(
    create_agent(AgentOptions(
        name="Researcher",
        instructions=(
            "You are a research assistant for a Lost podcast. "
            "Given a topic, return 2-3 paragraphs of factual background from the show. "
            "Be concise and accurate."
        ),
    )),
    id="researcher",
)


# Bridge 1: plain topic string → AgentExecutorRequest for the researcher
@executor(id="topic_to_request")
async def topic_to_request(topic: str, ctx: WorkflowContext[AgentExecutorRequest]) -> None:
    """Converts a plain topic string into an AgentExecutorRequest."""
    await ctx.send_message(
        AgentExecutorRequest(
            messages=[Message(role="user", contents=[topic])],
            should_respond=True,
        )
    )


# Bridge 2: researcher response → AgentExecutorRequest that gives Mara context
@executor(id="research_to_host")
async def research_to_host(
    response: AgentExecutorResponse, ctx: WorkflowContext[AgentExecutorRequest]
) -> None:
    """Packages research output as a contextual prompt for the host."""
    research_text = response.agent_run_response.text
    await ctx.send_message(
        AgentExecutorRequest(
            messages=[Message(
                role="user",
                contents=[
                    f"Here is research on today's topic:\n\n{research_text}\n\n"
                    "Based on this, what's your angle for the episode?"
                ]
            )],
            should_respond=True,
        )
    )


# 4-node research pipeline
research_pipeline = (
    WorkflowBuilder(start_executor=topic_to_request)
    .add_edge(topic_to_request, research_executor)
    .add_edge(research_executor, research_to_host)
    .add_edge(research_to_host, mara_executor)
    .add_edge(mara_executor, response_printer)
    .build()
)

In [ ]:
TOPIC = "The smoke monster — is it the island itself, or something else?"

print(f"Topic: {TOPIC}")
print("=" * 60)

events = await research_pipeline.run(TOPIC)
for output in events.get_outputs():
    print(output)

---
## 7. Conditional edges

Sometimes the right path through the graph depends on the data. You can attach a **condition function** to an edge — the framework only traverses that edge if the function returns `True` for the message the upstream executor sent.

```python
builder.add_edge(source, target, condition=my_condition_fn)
```

In the podcast context, Mara owns plot/mystery topics and Dev owns character psychology. The workflow below classifies the topic first, then routes to the right host.

```
topic (str)
    │
    ▼
topic_classifier  ──[PlotTopic]──────▶  plot_to_mara  ──▶  mara_executor  ──▶  response_printer
                  ──[CharacterTopic]──▶  char_to_dev   ──▶  dev_executor   ──▶  response_printer
```

By using **typed dataclasses** as the message output of `topic_classifier`, the condition functions become simple `isinstance` checks — readable and easy to extend.

In [ ]:
@dataclass
class PlotTopic:
    topic: str

@dataclass
class CharacterTopic:
    topic: str


_CHARACTER_KEYWORDS = [
    "jack", "locke", "sawyer", "kate", "hurley",
    "desmond", "ben", "richard", "character", "psychology",
]


@executor(id="topic_classifier")
async def topic_classifier(topic: str, ctx: WorkflowContext) -> None:
    """Classifies the topic and routes it to the appropriate host."""
    if any(kw in topic.lower() for kw in _CHARACTER_KEYWORDS):
        print("\u2192 CHARACTER topic — routing to Dev")
        await ctx.send_message(CharacterTopic(topic=topic))
    else:
        print("\u2192 PLOT/MYSTERY topic — routing to Mara")
        await ctx.send_message(PlotTopic(topic=topic))


# Bridge executors — each converts the typed topic into an AgentExecutorRequest
@executor(id="plot_to_mara")
async def plot_to_mara(pt: PlotTopic, ctx: WorkflowContext[AgentExecutorRequest]) -> None:
    await ctx.send_message(AgentExecutorRequest(
        messages=[Message(role="user", contents=[f"Your plot/mystery angle on: {pt.topic}"])],
        should_respond=True,
    ))

@executor(id="char_to_dev")
async def char_to_dev(ct: CharacterTopic, ctx: WorkflowContext[AgentExecutorRequest]) -> None:
    await ctx.send_message(AgentExecutorRequest(
        messages=[Message(role="user", contents=[f"Your character psychology angle on: {ct.topic}"])],
        should_respond=True,
    ))


# Condition functions — each receives the message the classifier sent
def is_plot_topic(msg: Any) -> bool:
    return isinstance(msg, PlotTopic)

def is_character_topic(msg: Any) -> bool:
    return isinstance(msg, CharacterTopic)


# Build the conditional workflow
conditional_workflow = (
    WorkflowBuilder(start_executor=topic_classifier)
    .add_edge(topic_classifier, plot_to_mara,  condition=is_plot_topic)
    .add_edge(topic_classifier, char_to_dev,   condition=is_character_topic)
    .add_edge(plot_to_mara, mara_executor)
    .add_edge(char_to_dev,  dev_executor)
    .add_edge(mara_executor, response_printer)
    .add_edge(dev_executor,  response_printer)
    .build()
)

In [ ]:
# A character-focused topic — should route to Dev
print("CHARACTER TOPIC")
print("=" * 60)
events = await conditional_workflow.run(
    "Locke's relationship with his father — how does it shape his arc on the island?"
)
for output in events.get_outputs():
    print(output)

In [ ]:
# A plot/mystery topic — should route to Mara
print("PLOT/MYSTERY TOPIC")
print("=" * 60)
events = await conditional_workflow.run(
    "Why does the island need to be protected, and from what exactly?"
)
for output in events.get_outputs():
    print(output)

---
## 8. Challenge: fan-out to both hosts in parallel

So far, each topic has gone to only one host. But a real podcast needs *both* perspectives. The graph API supports **fan-out**: if you add multiple edges from the same source executor, the framework sends the message to *all* connected targets and runs them in the same superstep — in parallel.

```
topic (str)
    │
    ▼
topic_to_request
    │
    ▼
research_executor
    │
    ▼
research_to_host  ──▶  mara_executor  ──▶  episode_brief
                  ──▶  dev_executor   ──▶  episode_brief
                        (parallel, same superstep)
```

Both hosts run simultaneously. Each one yields its output independently, so `events.get_outputs()` returns two items — one per host. Try changing `EPISODE_TOPIC` to something divisive and see how each host frames it differently.

In [ ]:
# Terminal executor shared by both hosts
@executor(id="episode_brief")
async def episode_brief(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Yields each host's response as a separate workflow output."""
    await ctx.yield_output(response.agent_run_response.text)


# Fan-out workflow: research_to_host sends the same request to BOTH hosts simultaneously
fanout_workflow = (
    WorkflowBuilder(start_executor=topic_to_request)
    .add_edge(topic_to_request, research_executor)
    .add_edge(research_executor, research_to_host)
    # Fan-out: two edges from the same source — both hosts run in parallel
    .add_edge(research_to_host, mara_executor)
    .add_edge(research_to_host, dev_executor)
    .add_edge(mara_executor, episode_brief)
    .add_edge(dev_executor,  episode_brief)
    .build()
)


EPISODE_TOPIC = "The numbers — 4 8 15 16 23 42. What do they really mean?"

print(f"Topic: {EPISODE_TOPIC}")
print("=" * 60)

events = await fanout_workflow.run(EPISODE_TOPIC)

for i, output in enumerate(events.get_outputs(), 1):
    print(f"\n--- Host {i} ---")
    print(output)

---
## Recap

Here's every pattern you used in this exercise:

```python
# 1. Class-based executor
class MyExecutor(Executor):
    @handler
    async def handle(self, message: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(message.upper())

# 2. Function-based executor
@executor(id="my_fn_executor")
async def my_fn_executor(message: str, ctx: WorkflowContext[Never, str]) -> None:
    await ctx.yield_output(message)

# 3. Wrap an AI agent as a graph node
agent_node = AgentExecutor(create_agent(AgentOptions(...)), id="my_agent")

# 4. Build the graph
workflow = (
    WorkflowBuilder(start_executor=first_node)
    .add_edge(first_node, second_node)               # direct edge
    .add_edge(second_node, path_a, condition=fn_a)   # conditional edge
    .add_edge(second_node, path_b, condition=fn_b)   # alternative path
    .add_edge(second_node, parallel_c)               # fan-out (multiple edges from same source)
    .add_edge(second_node, parallel_d)
    .build()
)

# 5. Run — non-streaming
events = await workflow.run(input_message)
outputs = events.get_outputs()          # list of ctx.yield_output(...) values

# 5. Run — streaming
async for event in workflow.run(input, stream=True):
    if event.type == "output":
        print(event.data)
```

**Next up — Exercise 4:** You'll use a workflow which uses the framework orchestration pattern [Group Chat](https://learn.microsoft.com/en-us/agent-framework/workflows/orchestrations/group-chat?pivots=programming-language-python) 